# USB microphone monitoring over Remote SSH

Capture the BY-V microphone attached to the remote machine and listen on **the computer running your browser**. Select **This computer (browser)**, then click **Start** to start a loopback streaming server. No audio is played on the remote machine in this mode.

1. In VS Code's **Ports** panel, choose **Forward a Port** and enter `8765` (or the server port selected below).
2. Open the player link printed below in your **local external browser**. If VS Code assigned a different local port, replace the port in the link and keep the `?token=...` part.
3. Click **Play** in that browser. Use its volume slider. Click **Stop** there to release the microphone, and **Stop** in this notebook to close the server.

The link contains a temporary access token. The server listens only on remote `127.0.0.1`; keep the SSH forwarding private. Closing the notebook tab does not stop its kernel or server. Audio is live PCM, not saved, and no ASR model is loaded. Network jitter can cause small playback gaps; this is monitoring, not synchronized audio transport.

For playback through speakers attached to the remote host, select **Remote machine speakers** instead. Use headphones to avoid feedback.

Install dependencies from the project terminal:

```bash
uv pip install --python .venv/bin/python -r requirements-audio-stream.txt
```

Select the project `.venv` kernel. On Linux, PortAudio must be available (`libportaudio2` on Debian/Ubuntu).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "speech_to_text").is_dir():
    raise RuntimeError("Run this notebook from the project root.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project-local .venv kernel before continuing.")

import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display
from html import escape
from speech_to_text.monitor import MicrophoneMonitor
from speech_to_text.browser_monitor import BrowserMicrophoneMonitor

print(sd.query_devices())
print("Device indices above are PortAudio indices, not ALSA card numbers.")

   0 HDA Intel PCH: ALC888-VD Analog (hw:0,0), ALSA (2 in, 2 out)
   1 HDA Intel PCH: HDMI 0 (hw:0,3), ALSA (0 in, 8 out)
   2 HDA Intel PCH: HDMI 1 (hw:0,7), ALSA (0 in, 8 out)
   3 HDA Intel PCH: HDMI 2 (hw:0,8), ALSA (0 in, 8 out)
   4 HDA Intel PCH: HDMI 3 (hw:0,9), ALSA (0 in, 8 out)
   5 BY-V: USB Audio (hw:1,0), ALSA (2 in, 0 out)
   6 sysdefault, ALSA (128 in, 128 out)
   7 front, ALSA (0 in, 2 out)
   8 surround40, ALSA (0 in, 2 out)
   9 surround51, ALSA (0 in, 2 out)
  10 surround71, ALSA (0 in, 2 out)
  11 hdmi, ALSA (0 in, 8 out)
  12 pipewire, ALSA (64 in, 64 out)
  13 dmix, ALSA (0 in, 2 out)
* 14 default, ALSA (64 in, 64 out)
Device indices above are PortAudio indices, not ALSA card numbers.


In [2]:
# Release streams and the HTTP port before recreating controls.
for name in ("monitor", "browser_monitor"):
    previous_monitor = globals().get(name)
    if previous_monitor is not None:
        previous_monitor.close()
previous_controls = globals().get("controls")
if previous_controls is not None:
    for control in previous_controls.children:
        control.close()
    previous_controls.close()
previous_status = globals().get("status")
if previous_status is not None:
    previous_status.close()

monitor = MicrophoneMonitor()
browser_monitor = BrowserMicrophoneMonitor()
devices = sd.query_devices()
input_options = [(f"{index}: {info['name']}", index)
                 for index, info in enumerate(devices)
                 if info["max_input_channels"] > 0]
output_options = [("System default output", None)] + [
    (f"{index}: {info['name']}", index)
    for index, info in enumerate(devices) if info["max_output_channels"] > 0
]
preferred_input = next((index for label, index in input_options if "BY-V" in label), None)
mic = widgets.Dropdown(options=input_options, description="Microphone:",
                       layout=widgets.Layout(width="650px"))
if preferred_input is not None:
    mic.value = preferred_input
target = widgets.Dropdown(options=[("This computer (browser)", "browser"),
                                  ("Remote machine speakers", "remote")],
                          description="Listen on:", layout=widgets.Layout(width="650px"))
output = widgets.Dropdown(options=output_options, description="Remote out:",
                          layout=widgets.Layout(width="650px"))
port = widgets.IntText(value=8765, description="Server port:")
sample_rate = widgets.Dropdown(options=[48000, 44100], value=48000, description="Rate (Hz):")
gain = widgets.FloatSlider(value=0.1, min=0, max=1, step=0.05, description="Remote gain:")
start_button = widgets.Button(description="Start", button_style="success", disabled=not input_options)
stop_button = widgets.Button(description="Stop", button_style="warning", disabled=True)
player_link = widgets.HTML()
status = widgets.Output()


def set_running(running):
    start_button.disabled = running or not input_options
    stop_button.disabled = not running
    for control in (mic, target, sample_rate):
        control.disabled = running
    output.disabled = gain.disabled = running or target.value == "browser"
    port.disabled = running or target.value != "browser"


def start_monitoring(_):
    with status:
        status.clear_output(wait=True)
        try:
            rate = sample_rate.value
            assert isinstance(rate, int), "Select a sample rate."
            if target.value == "browser":
                if not 1 <= port.value <= 65535:
                    raise ValueError("Server port must be between 1 and 65535.")
                url = browser_monitor.start(mic.value, port=port.value, samplerate=rate)
                player_link.value = f'<a href="{escape(url)}" target="_blank" rel="noreferrer">Open player in your local browser</a>'
                print(f"Forward remote port {port.value} in VS Code's Ports panel.")
                print(f"Open in your LOCAL external browser, then click Play: {url}")
                print("If the forwarded local port differs, replace only the port in this URL.")
                print("Use the browser player's volume slider. Waiting for a listener.")
            else:
                monitor.start(mic.value, output.value, rate, gain.value)
                print("Monitoring through the remote machine's speakers.")
            set_running(True)
        except BaseException as error:
            try:
                monitor.close()
            finally:
                browser_monitor.close()
                player_link.value = ""
                set_running(False)
            print(f"Could not start monitoring: {error}")
            if not isinstance(error, Exception):
                raise


def stop_monitoring(_):
    active_monitor = browser_monitor if target.value == "browser" else monitor
    try:
        try:
            monitor.close()
        finally:
            browser_monitor.close()
    finally:
        player_link.value = ""
        set_running(False)
    with status:
        status.clear_output(wait=True)
        print(f"Stopped. Peak input amplitude: {active_monitor.peak:.4f}")
        if active_monitor.last_status:
            print(f"Last audio status: {active_monitor.last_status}")


target.observe(lambda change: set_running(False), names="value")
start_button.on_click(start_monitoring)
stop_button.on_click(stop_monitoring)
set_running(False)
controls = widgets.VBox([mic, target, output, port, sample_rate, gain,
                         start_button, stop_button, player_link])
display(controls, status)
if preferred_input is None:
    print("BY-V was not found. Select an available microphone or reconnect and rerun this cell.")

Output()

If Start reports an unsupported sample rate, stop and try the other rate. Browser mode requires only microphone input support; remote-speaker mode requires both devices to support the same rate. Stop and rerun the device and controls cells after reconnecting hardware.

If the player cannot connect, verify the notebook server is running and the remote port is forwarded to the local port in the URL. Use a local external browser so playback occurs on your computer. Browser playback requires clicking Play. Only one listener is supported at a time; Stop the previous player before reconnecting.

Browser Stop/disconnect releases microphone capture. Notebook Stop also closes the server and its port. Rerunning the controls cell closes both old modes. An idle kernel interrupt does not necessarily stop background audio or HTTP threads; click Stop or run the cleanup cell below. Kernel shutdown releases device handles.

In [3]:
# Explicit cleanup; safe to run after stopping or interrupting work.
try:
    if "monitor" in globals():
        monitor.close()
finally:
    if "browser_monitor" in globals():
        browser_monitor.close()
if "player_link" in globals():
    player_link.value = ""
if "set_running" in globals():
    set_running(False)
print("Audio streams and browser server closed.")

Audio streams and browser server closed.
